<a href="https://colab.research.google.com/github/jeancarlos2015/ExerciciosTPA/blob/master/metricas_utilizando_rede_petri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pm4py pandas


In [27]:
# Adicione esta função modular antes do passo de avaliação
def normalizar_rotulos_rede(net):
    """
    Remove distinções de maiúsculas/minúsculas e espaços extras nas transições da IA
    para evitar notas zero por diferenças puramente textuais.
    """
    for transition in net.transitions:
        if transition.label:
            # Transforma em minúsculo, remove espaços nas pontas e padroniza
            transition.label = " ".join(transition.label.strip().lower().split())
    return net


In [50]:
# import pm4py
# from typing import Tuple, Dict, Any

# # Importações dos submódulos internos e estáveis de algoritmos
# from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
# from pm4py.algo.evaluation.precision import algorithm as precision_evaluator

# def carregar_e_converter_modelo(caminho_bpmn: str) -> Tuple[Any, Any, Any]:
#     """
#     Passo 1 & 3: Carrega o arquivo BPMN e converte para componentes de Rede de Petri.
#     """
#     print(f" -> Processando arquivo: {caminho_bpmn}")
#     bpmn_model = pm4py.read_bpmn(caminho_bpmn)
#     return pm4py.convert_to_petri_net(bpmn_model)


# def executar_play_out_gabarito(net_ref: Any, im_ref: Any, fm_ref: Any) -> Any:
#     """
#     Passo 2: Executa o Play-Out matemático sobre a rede de referência para gerar as instâncias de teste.
#     """
#     print("[Passo 2] Executando Play-Out direto sobre o gabarito...")
#     log = pm4py.play_out(net_ref, im_ref, fm_ref)
#     print(f" -> Log Sintético gerado com sucesso! Total de casos simulados: {len(log)}")
#     return log


# def normalizar_modelo_e_log(net: Any, log: Any = None) -> Any:
#     """
#     Ação de Tolerância: Remove distinções de maiúsculas/minúsculas, espaços extras
#     e quebras de linha tanto na Rede de Petri quanto no Log de Eventos.
#     """
#     # 1. Normaliza os rótulos das transições na Rede de Petri
#     for transition in net.transitions:
#         if transition.label:
#             # Converte para minúsculo e limpa espaços extras/quebras de linha
#             transition.label = " ".join(transition.label.strip().lower().split())

#     # 2. Normaliza os nomes das atividades dentro do log (se ele for fornecido)
#     if log is not None:
#         for trace in log:
#             for event in trace:
#                 if 'concept:name' in event:
#                     event['concept:name'] = " ".join(event['concept:name'].strip().lower().split())
#         return net, log

#     return net


# def calcular_simplicidade_estrutural(net: Any) -> float:
#     """
#     Métrica de Simplicidade baseada no Process Mining Handbook.
#     Avalia: Tamanho Absoluto, CFC (Cardoso) e Proporção de Transições Silenciosas.
#     """
#     # 1. Componente de Tamanho Absoluto
#     lugares = len(net.places)
#     transicoes = len(net.transitions)
#     arcos = len(net.arcs)

#     tamanho_total = lugares + transicoes + arcos
#     componente_tamanho = 1.0 / (1.0 + tamanho_total) if tamanho_total > 0 else 0.0

#     # 2. Componente de Complexidade do Fluxo de Controle (CFC baseado em Cardoso)
#     cfc_total = 0.0
#     for t in net.transitions:
#         arcos_saida = len(t.out_arcs)
#         if arcos_saida > 1:
#             if t.label is None:
#                 cfc_total += arcos_saida  # Comportamento XOR / Desvios Lógicos
#             else:
#                 cfc_total += 1.0          # Paralelismo implícito

#     componente_fluxo = 1.0 / (1.0 + cfc_total)

#     # 3. Componente de Transições Silenciosas (Tau steps / Gateways Invisíveis)
#     num_reais = sum(1 for t in net.transitions if t.label is not None)
#     num_silenciosas = sum(1 for t in net.transitions if t.label is None)

#     componente_silenciosa = num_reais / (num_reais + num_silenciosas) if (num_reais + num_silenciosas) > 0 else 0.0

#     # 4. Score Final Ponderado (Pesos: 20% Tamanho, 40% Fluxo, 40% Silenciosa)
#     return (componente_tamanho * 0.2) + (componente_fluxo * 0.4) + (componente_silenciosa * 0.4)


# def calcular_metricas_conformance(log: Any, net_ai: Any, im_ai: Any, fm_ai: Any) -> Tuple[float, float, str]:
#     """
#     Passo 4: Realiza os cálculos de Fitness e Precision com fallback resiliente para Redes Não-Sãs.
#     """
#     print("[Passo 4] Calculando Métricas de Qualidade...")

#     try:
#         # Tentativa padrão via Alinhamentos (Matriz Ótima)
#         metodo = "Alinhamentos (Matriz Ótima A*)"

#         fitness_results = pm4py.fitness_alignments(log, net_ai, im_ai, fm_ai)
#         fitness_score = fitness_results.get('log_fitness', 0.0)

#         precision_results = pm4py.precision_alignments(log, net_ai, im_ai, fm_ai)
#         if isinstance(precision_results, dict):
#             precision_score = precision_results.get('percentage_of_precision', precision_results.get('precision', 0.0))
#         else:
#             precision_score = precision_results

#     except Exception as e:
#         # Fallback caso a Rede da IA contenha Deadlocks ou Loops malformados (Não-Sound)
#         print(f"\n[Aviso] Alinhamentos falharam (Rede Não-Sã): {e}")
#         print("-> Alterando automaticamente para a abordagem Token-Based Replay...")

#         metodo = "Token-Based Replay (Contagem de Fichas)"

#         # 1. Cálculo de Fitness usando a nomenclatura correta (TOKEN_BASED)
#         fitness_results = replay_fitness_evaluator.apply(
#             log, net_ai, im_ai, fm_ai,
#             variant=replay_fitness_evaluator.Variants.TOKEN_BASED
#         )
#         fitness_score = fitness_results.get('log_fitness', 0.0)

#         # 2. Cálculo de Precision tolerante via variante ETCONFORMANCE_TOKEN
#         precision_score = precision_evaluator.apply(
#             log, net_ai, im_ai, fm_ai,
#             variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN
#         )
#         if isinstance(precision_score, dict):
#             precision_score = precision_score.get('precision', 0.0)

#     return fitness_score, precision_score, metodo


# def exibir_relatorio_resultados(fitness_score: float, precision_score: float, metodo: str) -> None:
#     """
#     Apresentação formatada das métricas e diagnósticos estruturais para o artigo.
#     """
#     print("\n" + "="*40)
#     print("         RESULTADO DA AVALIAÇÃO DA IA       ")
#     print("="*40)
#     print(f"MÉTODO USADO:      {metodo}")
#     print(f"FITNESS (RECALL):  {fitness_score * 100:.2f}%")
#     print(f"PRECISION:         {precision_score * 100:.2f}%")
#     print("="*40)

#     if fitness_score >= 0.99 and precision_score >= 0.99:
#         print("Resultado: O modelo da IA é semanticamente perfeito em relação ao gabarito!")
#     else:
#         if "Token-Based" in metodo:
#             print("\n[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:")
#             print("- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.")
#             print("- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).")

#         print("\nAnálise Comportamental:")
#         if fitness_score < 1.0:
#             print(f"- O modelo da IA bloqueia ou omite {((1.0 - fitness_score) * 100):.1f}% dos caminhos legítimos do gabarito.")
#         if precision_score < 1.0:
#             print(f"- O modelo da IA está frouxo, permitindo {((1.0 - precision_score) * 100):.1f}% de caminhos 'fantasmas' inválidos.")


# def run_evaluation_pipeline(bpmn_reference_path: str, bpmn_ai_path: str) -> Dict[str, Any]:
#     """
#     Função Orquestradora do Pipeline Completo.
#     """
#     print("=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===")

#     # Passo 1: Importar e preparar o Gabarito
#     print("\n[Passo 1] Importando o BPMN de Referência...")
#     net_ref, im_ref, fm_ref = carregar_e_converter_modelo(bpmn_reference_path)

#     # [CAMADA ADICIONADA]: Normaliza a rede de referência antes do Play-Out
#     net_ref = normalizar_modelo_e_log(net_ref)

#     # Passo 2: Geração do Log Sintético
#     synthetic_log = executar_play_out_gabarito(net_ref, im_ref, fm_ref)

#     # Passo 3: Importar e preparar o Modelo da IA
#     print("\n[Passo 3] Importando e convertendo o modelo da IA...")
#     net_ai, im_ai, fm_ai = carregar_e_converter_modelo(bpmn_ai_path)

#     # [CAMADA ADICIONADA]: Normaliza a Rede da IA e o Log Sintético para garantir pareamento textual idêntico
#     net_ai, synthetic_log = normalizar_modelo_e_log(net_ai, synthetic_log)

#     # Passo 4: Executar Conformance Checking
#     fitness, precision, metodo_utilizado = calcular_metricas_conformance(synthetic_log, net_ai, im_ai, fm_ai)

#     # Apresentação Final
#     exibir_relatorio_resultados(fitness, precision, metodo_utilizado)

#     return {"fitness": fitness, "precision": precision, "metodo": metodo_utilizado}

import pm4py
from typing import Tuple, Dict, Any

# Importações dos submódulos internos e estáveis de algoritmos para o Conformance
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator

def carregar_e_converter_modelo(caminho_bpmn: str) -> Tuple[Any, Any, Any]:
    """
    Passo 1 & 3: Carrega o arquivo BPMN e converte para componentes de Rede de Petri.
    """
    print(f" -> Processando arquivo: {caminho_bpmn}")
    bpmn_model = pm4py.read_bpmn(caminho_bpmn)
    return pm4py.convert_to_petri_net(bpmn_model)


def executar_play_out_gabarito(net_ref: Any, im_ref: Any, fm_ref: Any) -> Any:
    """
    Passo 2: Executa o Play-Out matemático sobre a rede de referência para gerar as instâncias de teste.
    """
    print("[Passo 2] Executando Play-Out direto sobre o gabarito...")
    log = pm4py.play_out(net_ref, im_ref, fm_ref)
    print(f" -> Log Sintético gerado com sucesso! Total de casos simulados: {len(log)}")
    return log


def normalizar_modelo_e_log(net: Any, log: Any = None) -> Any:
    """
    Ação de Tolerância: Remove distinções de maiúsculas/minúsculas, espaços extras
    e quebras de linha tanto na Rede de Petri quanto no Log de Eventos.
    """
    # 1. Normaliza os rótulos das transições na Rede de Petri
    for transition in net.transitions:
        if transition.label:
            # Converte para minúsculo e limpa espaços extras/quebras de linha
            transition.label = " ".join(transition.label.strip().lower().split())

    # 2. Normaliza os nomes das atividades dentro do log (se ele for fornecido)
    if log is not None:
        for trace in log:
            for event in trace:
                if 'concept:name' in event:
                    event['concept:name'] = " ".join(event['concept:name'].strip().lower().split())
        return net, log

    return net


def calcular_simplicidade_estrutural(net: Any) -> float:
    """
    Métrica de Simplicidade baseada no Process Mining Handbook.
    Avalia: Tamanho Absoluto, CFC (Cardoso) e Proporção de Transições Silenciosas.
    """
    # 1. Componente de Tamanho Absoluto
    lugares = len(net.places)
    transicoes = len(net.transitions)
    arcos = len(net.arcs)

    tamanho_total = lugares + transicoes + arcos
    componente_tamanho = 1.0 / (1.0 + tamanho_total) if tamanho_total > 0 else 0.0

    # 2. Componente de Complexidade do Fluxo de Controle (CFC baseado em Cardoso)
    cfc_total = 0.0
    for t in net.transitions:
        arcos_saida = len(t.out_arcs)
        if arcos_saida > 1:
            if t.label is None:
                cfc_total += arcos_saida  # Comportamento XOR / Desvios Lógicos
            else:
                cfc_total += 1.0          # Paralelismo implícito

    componente_fluxo = 1.0 / (1.0 + cfc_total)

    # 3. Componente de Transições Silenciosas (Tau steps / Gateways Invisíveis)
    num_reais = sum(1 for t in net.transitions if t.label is not None)
    num_silenciosas = sum(1 for t in net.transitions if t.label is None)

    componente_silenciosa = num_reais / (num_reais + num_silenciosas) if (num_reais + num_silenciosas) > 0 else 0.0

    # 4. Score Final Ponderado (Pesos: 20% Tamanho, 40% Fluxo, 40% Silenciosa)
    return (componente_tamanho * 0.2) + (componente_fluxo * 0.4) + (componente_silenciosa * 0.4)


def calcular_metricas_conformance(log: Any, net_ai: Any, im_ai: Any, fm_ai: Any) -> Tuple[float, float, str]:
    """
    Passo 4: Realiza os cálculos de Fitness e Precision com fallback resiliente para Redes Não-Sãs.
    """
    print("[Passo 4] Calculando Métricas de Qualidade...")

    try:
        # Tentativa padrão via Alinhamentos (Matriz Ótima)
        metodo = "Alinhamentos (Matriz Ótima A*)"

        fitness_results = pm4py.fitness_alignments(log, net_ai, im_ai, fm_ai)
        fitness_score = fitness_results.get('log_fitness', 0.0)

        precision_results = pm4py.precision_alignments(log, net_ai, im_ai, fm_ai)
        if isinstance(precision_results, dict):
            precision_score = precision_results.get('percentage_of_precision', precision_results.get('precision', 0.0))
        else:
            precision_score = precision_results

    except Exception as e:
        # Fallback caso a Rede da IA contenha Deadlocks ou Loops malformados (Não-Sound)
        print(f"\n[Aviso] Alinhamentos falharam (Rede Não-Sã): {e}")
        print("-> Alterando automaticamente para a abordagem Token-Based Replay...")

        metodo = "Token-Based Replay (Contagem de Fichas)"

        # 1. Cálculo de Fitness usando a nomenclatura correta (TOKEN_BASED)
        fitness_results = replay_fitness_evaluator.apply(
            log, net_ai, im_ai, fm_ai,
            variant=replay_fitness_evaluator.Variants.TOKEN_BASED
        )
        fitness_score = fitness_results.get('log_fitness', 0.0)

        # 2. Cálculo de Precision tolerante via variante ETCONFORMANCE_TOKEN
        precision_score = precision_evaluator.apply(
            log, net_ai, im_ai, fm_ai,
            variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN
        )
        if isinstance(precision_score, dict):
            precision_score = precision_score.get('precision', 0.0)

    return fitness_score, precision_score, metodo


import pm4py
from typing import Any

def calcular_generalizacao_comportamental(log: Any, net: Any, im: Any, fm_ai: Any) -> float:
    """
    Calcula a Generalização de um modelo de processo segundo a metodologia
    do Process Mining Handbook, avaliando se a Rede de Petri evita o overfitting.

    Returns:
        float: Score de Generalização entre 0.0 (overfitting estrito) e 1.0 (generalista ideal).
    """
    print(" -> Calculando a Métrica de Generalização...")

    # Importa o módulo oficial de avaliação de generalização do PM4Py
    from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator

    try:
        # O PM4Py calcula a generalização simulando a passagem do log sintético pela rede da IA.
        # Caso a rede possua deadlocks que travem o motor probabilístico, capturamos na exceção.
        score_generalizacao = generalization_evaluator.apply(log, net, im, fm_ai)

    except Exception as e:
        print(f" -> [Aviso] Falha estrutural no cálculo probabilístico de Generalização: {e}")
        print(" -> Aplicando fallback analítico (Considerando Overfitting por travamento de fluxo).")
        # Se o modelo gerado pela IA estiver tão quebrado que não permite que os lugares sejam visitados,
        # sua capacidade de generalizar o comportamento futuro cai para o piso mínimo.
        score_generalizacao = 0.0

    return score_generalizacao


def exibir_relatorio_resultados(fitness_score: float, precision_score: float, simplicidade_ref: float, simplicidade_ai: float, generalizacao_score: float, metodo: str) -> None:
    """
    Apresentação formatada das métricas e diagnósticos estruturais para o artigo.
    """
    print("\n" + "="*40)
    print("         RESULTADO DA AVALIAÇÃO DA IA       ")
    print("="*40)
    print(f"MÉTODO USADO:      {metodo}")
    print(f"FITNESS (RECALL):  {fitness_score * 100:.2f}%")
    print(f"PRECISION:         {precision_score * 100:.2f}%")
    print(f"GENERALIZAÇÃO:     {generalizacao_score * 100:.2f}%")
    print("-"*40)
    print(f"SIMPLICIDADE GABARITO: {simplicidade_ref * 100:.2f}%")
    print(f"SIMPLICIDADE IA (QWEN):{simplicidade_ai * 100:.2f}%")
    print("="*40)

    if fitness_score >= 0.99 and precision_score >= 0.99:
        print("Resultado: O modelo da IA é semanticamente perfeito em relação ao gabarito!")
    else:
        if "Token-Based" in metodo:
            print("\n[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:")
            print("- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.")
            print("- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).")

        print("\nAnálise Comportamental:")
        if fitness_score < 1.0:
            print(f"- O modelo da IA bloqueia ou omite {((1.0 - fitness_score) * 100):.1f}% dos caminhos legítimos do gabarito.")
        if precision_score < 1.0:
            print(f"- O modelo da IA está frouxo, permitindo {((1.0 - precision_score) * 100):.1f}% de caminhos 'fantasmas' inválidos.")


def run_evaluation_pipeline(bpmn_reference_path: str, bpmn_ai_path: str) -> Dict[str, Any]:
    """
    Função Orquestradora do Pipeline Completo do Quadrante de Qualidade.
    """
    print("=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===")

    # Passo 1: Importar e preparar o Gabarito
    print("\n[Passo 1] Importando o BPMN de Referência...")
    net_ref, im_ref, fm_ref = carregar_e_converter_modelo(bpmn_reference_path)

    # [CAMADA ADICIONADA]: Normaliza a rede de referência antes do Play-Out
    net_ref = normalizar_modelo_e_log(net_ref)

    # Passo 2: Geração do Log Sintético
    synthetic_log = executar_play_out_gabarito(net_ref, im_ref, fm_ref)

    # Passo 3: Importar e preparar o Modelo da IA
    print("\n[Passo 3] Importando e convertendo o modelo da IA...")
    net_ai, im_ai, fm_ai = carregar_e_converter_modelo(bpmn_ai_path)

    # [CAMADA ADICIONADA]: Normaliza a Rede da IA e o Log Sintético para garantir pareamento textual idêntico
    net_ai, synthetic_log = normalizar_modelo_e_log(net_ai, synthetic_log)

    # Passo 4: Executar Conformance Checking (Fitness e Precision)
    fitness, precision, metodo_utilizado = calcular_metricas_conformance(synthetic_log, net_ai, im_ai, fm_ai)

    # Passo 5: Executar Cálculo de Simplicidade Reutilizando as Redes de Petri carregadas
    simplicidade_ref = calcular_simplicidade_estrutural(net_ref)
    simplicidade_ai = calcular_simplicidade_estrutural(net_ai)

    # [CAMADA ADICIONADA]: Passo 6: Executar Cálculo de Generalização para a IA
    generalizacao_ai = calcular_generalizacao_comportamental(synthetic_log, net_ai, im_ai, fm_ai)

    # Apresentação Final repassando as métricas do quadrante atualizado
    exibir_relatorio_resultados(
        fitness,
        precision,
        simplicidade_ref,
        simplicidade_ai,
        generalizacao_ai,
        metodo_utilizado
    )

    return {
        "fitness": fitness,
        "precision": precision,
        "generalization": generalizacao_ai,
        "simplicidade_ref": simplicidade_ref,
        "simplicidade_ai": simplicidade_ai,
        "metodo": metodo_utilizado
    }



In [51]:
if __name__ == "__main__":
    gabarito_path = "xml_referencia.bpmn"
    modelo_ia_path = "xml_gerado_qwen_rede_petri.bpmn"

    try:
        run_evaluation_pipeline(gabarito_path, modelo_ia_path)
    except FileNotFoundError:
        print(f"\n[Erro] Certifique-se de salvar os arquivos '{gabarito_path}' e '{modelo_ia_path}' no mesmo diretório.")

=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===

[Passo 1] Importando o BPMN de Referência...
 -> Processando arquivo: xml_referencia.bpmn
[Passo 2] Executando Play-Out direto sobre o gabarito...
 -> Log Sintético gerado com sucesso! Total de casos simulados: 1000

[Passo 3] Importando e convertendo o modelo da IA...
 -> Processando arquivo: xml_gerado_qwen_rede_petri.bpmn
[Passo 4] Calculando Métricas de Qualidade...

[Aviso] Alinhamentos falharam (Rede Não-Sã): trying to apply alignments on a Petri net that is not a easy sound net!!
-> Alterando automaticamente para a abordagem Token-Based Replay...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

 -> Calculando a Métrica de Generalização...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]


         RESULTADO DA AVALIAÇÃO DA IA       
MÉTODO USADO:      Token-Based Replay (Contagem de Fichas)
FITNESS (RECALL):  83.33%
PRECISION:         100.00%
GENERALIZAÇÃO:     35.21%
----------------------------------------
SIMPLICIDADE GABARITO: 56.43%
SIMPLICIDADE IA (QWEN):56.76%

[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:
- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.
- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).

Análise Comportamental:
- O modelo da IA bloqueia ou omite 16.7% dos caminhos legítimos do gabarito.


In [52]:
if __name__ == "__main__":
    gabarito_path = "xml_referencia.bpmn"
    modelo_ia_path = "xml_gerado gemma_rede_petri.bpmn"

    try:
        run_evaluation_pipeline(gabarito_path, modelo_ia_path)
    except FileNotFoundError:
        print(f"\n[Erro] Certifique-se de salvar os arquivos '{gabarito_path}' e '{modelo_ia_path}' no mesmo diretório.")

=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===

[Passo 1] Importando o BPMN de Referência...
 -> Processando arquivo: xml_referencia.bpmn
[Passo 2] Executando Play-Out direto sobre o gabarito...
 -> Log Sintético gerado com sucesso! Total de casos simulados: 1000

[Passo 3] Importando e convertendo o modelo da IA...
 -> Processando arquivo: xml_gerado gemma_rede_petri.bpmn
[Passo 4] Calculando Métricas de Qualidade...

[Aviso] Alinhamentos falharam (Rede Não-Sã): trying to apply alignments on a Petri net that is not a easy sound net!!
-> Alterando automaticamente para a abordagem Token-Based Replay...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

 -> Calculando a Métrica de Generalização...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]


         RESULTADO DA AVALIAÇÃO DA IA       
MÉTODO USADO:      Token-Based Replay (Contagem de Fichas)
FITNESS (RECALL):  83.33%
PRECISION:         100.00%
GENERALIZAÇÃO:     35.21%
----------------------------------------
SIMPLICIDADE GABARITO: 56.43%
SIMPLICIDADE IA (QWEN):56.76%

[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:
- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.
- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).

Análise Comportamental:
- O modelo da IA bloqueia ou omite 16.7% dos caminhos legítimos do gabarito.


In [53]:
if __name__ == "__main__":
    gabarito_path = "xml_referencia.bpmn"
    modelo_ia_path = "xml_gerado_llama_rede_petri.bpmn"

    try:
        run_evaluation_pipeline(gabarito_path, modelo_ia_path)
    except FileNotFoundError:
        print(f"\n[Erro] Certifique-se de salvar os arquivos '{gabarito_path}' e '{modelo_ia_path}' no mesmo diretório.")

=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===

[Passo 1] Importando o BPMN de Referência...
 -> Processando arquivo: xml_referencia.bpmn
[Passo 2] Executando Play-Out direto sobre o gabarito...
 -> Log Sintético gerado com sucesso! Total de casos simulados: 1000

[Passo 3] Importando e convertendo o modelo da IA...
 -> Processando arquivo: xml_gerado_llama_rede_petri.bpmn
[Passo 4] Calculando Métricas de Qualidade...

[Aviso] Alinhamentos falharam (Rede Não-Sã): trying to apply alignments on a Petri net that is not a easy sound net!!
-> Alterando automaticamente para a abordagem Token-Based Replay...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

 -> Calculando a Métrica de Generalização...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]


         RESULTADO DA AVALIAÇÃO DA IA       
MÉTODO USADO:      Token-Based Replay (Contagem de Fichas)
FITNESS (RECALL):  49.88%
PRECISION:         100.00%
GENERALIZAÇÃO:     35.21%
----------------------------------------
SIMPLICIDADE GABARITO: 56.43%
SIMPLICIDADE IA (QWEN):76.77%

[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:
- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.
- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).

Análise Comportamental:
- O modelo da IA bloqueia ou omite 50.1% dos caminhos legítimos do gabarito.


In [54]:
import pandas as pd

def evaluate_models_and_display_table():
    results = []

    # Evaluate Qwen model
    print("\n--- Avaliando Modelo Qwen ---")
    qwen_gabarito_path = "xml_referencia.bpmn"
    qwen_modelo_ia_path = "xml_gerado_qwen_rede_petri.bpmn"
    try:
        qwen_results = run_evaluation_pipeline(qwen_gabarito_path, qwen_modelo_ia_path)
        results.append({
            "Modelo": "Qwen",
            "Fitness (Recall)": f"{qwen_results['fitness'] * 100:.2f}%",
            "Precision": f"{qwen_results['precision'] * 100:.2f}%",
            "Generalização": f"{qwen_results['generalization'] * 100:.2f}%",
            "Simplicidade (Ref)": f"{qwen_results['simplicidade_ref'] * 100:.2f}%",
            "Simplicidade (AI)": f"{qwen_results['simplicidade_ai'] * 100:.2f}%",
            "Método": qwen_results['metodo']
        })
    except FileNotFoundError:
        print(f"[Erro] Arquivos para o modelo Qwen não encontrados: '{qwen_gabarito_path}' ou '{qwen_modelo_ia_path}'. Pulando avaliação.")
        results.append({
            "Modelo": "Qwen",
            "Fitness (Recall)": "N/A",
            "Precision": "N/A",
            "Generalização": "N/A",
            "Simplicidade (Ref)": "N/A",
            "Simplicidade (AI)": "N/A",
            "Método": "Erro: Arquivo não encontrado"
        })

    # Evaluate Llama model
    print("\n--- Avaliando Modelo Llama ---")
    llama_gabarito_path = "xml_referencia.bpmn"
    llama_modelo_ia_path = "xml_gerado_llama_rede_petri.bpmn"
    try:
        llama_results = run_evaluation_pipeline(llama_gabarito_path, llama_modelo_ia_path)
        results.append({
            "Modelo": "Llama",
            "Fitness (Recall)": f"{llama_results['fitness'] * 100:.2f}%",
            "Precision": f"{llama_results['precision'] * 100:.2f}%",
            "Generalização": f"{llama_results['generalization'] * 100:.2f}%",
            "Simplicidade (Ref)": f"{llama_results['simplicidade_ref'] * 100:.2f}%",
            "Simplicidade (AI)": f"{llama_results['simplicidade_ai'] * 100:.2f}%",
            "Método": llama_results['metodo']
        })
    except FileNotFoundError:
        print(f"[Erro] Arquivos para o modelo Llama não encontrados: '{llama_gabarito_path}' ou '{llama_modelo_ia_path}'. Pulando avaliação.")
        results.append({
            "Modelo": "Llama",
            "Fitness (Recall)": "N/A",
            "Precision": "N/A",
            "Generalização": "N/A",
            "Simplicidade (Ref)": "N/A",
            "Simplicidade (AI)": "N/A",
            "Método": "Erro: Arquivo não encontrado"
        })

    # Evaluate Gemma model
    print("\n--- Avaliando Modelo Gemma ---")
    gemma_gabarito_path = "xml_referencia.bpmn"
    gemma_modelo_ia_path = "xml_gerado gemma_rede_petri.bpmn"
    try:
        gemma_results = run_evaluation_pipeline(gemma_gabarito_path, gemma_modelo_ia_path)
        results.append({
            "Modelo": "Gemma",
            "Fitness (Recall)": f"{gemma_results['fitness'] * 100:.2f}%",
            "Precision": f"{gemma_results['precision'] * 100:.2f}%",
            "Generalização": f"{gemma_results['generalization'] * 100:.2f}%",
            "Simplicidade (Ref)": f"{gemma_results['simplicidade_ref'] * 100:.2f}%",
            "Simplicidade (AI)": f"{gemma_results['simplicidade_ai'] * 100:.2f}%",
            "Método": gemma_results['metodo']
        })
    except FileNotFoundError:
        print(f"[Erro] Arquivos para o modelo Gemma não encontrados: '{gemma_gabarito_path}' ou '{gemma_modelo_ia_path}'. Pulando avaliação.")
        results.append({
            "Modelo": "Gemma",
            "Fitness (Recall)": "N/A",
            "Precision": "N/A",
            "Generalização": "N/A",
            "Simplicidade (Ref)": "N/A",
            "Simplicidade (AI)": "N/A",
            "Método": "Erro: Arquivo não encontrado"
        })

    # Create and display DataFrame
    df_results = pd.DataFrame(results)
    print("\n--- Resumo dos Resultados --- ")
    display(df_results)

# Call the function to run evaluations and display the table
evaluate_models_and_display_table()



--- Avaliando Modelo Qwen ---
=== INICIANDO PIPELINE DE CONFORMANCE CHECKING ===

[Passo 1] Importando o BPMN de Referência...
 -> Processando arquivo: xml_referencia.bpmn
[Passo 2] Executando Play-Out direto sobre o gabarito...
 -> Log Sintético gerado com sucesso! Total de casos simulados: 1000

[Passo 3] Importando e convertendo o modelo da IA...
 -> Processando arquivo: xml_gerado_qwen_rede_petri.bpmn
[Passo 4] Calculando Métricas de Qualidade...

[Aviso] Alinhamentos falharam (Rede Não-Sã): trying to apply alignments on a Petri net that is not a easy sound net!!
-> Alterando automaticamente para a abordagem Token-Based Replay...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

 -> Calculando a Métrica de Generalização...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

...

replaying log with TBR, completed traces ::   0%|          | 0/5 [00:00<?, ?it/s]

 -> Calculando a Métrica de Generalização...


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]


         RESULTADO DA AVALIAÇÃO DA IA       
MÉTODO USADO:      Token-Based Replay (Contagem de Fichas)
FITNESS (RECALL):  83.33%
PRECISION:         100.00%
GENERALIZAÇÃO:     35.21%
----------------------------------------
SIMPLICIDADE GABARITO: 56.43%
SIMPLICIDADE IA (QWEN):56.76%

[DIAGNÓSTICO CRÍTICO PARA O SEU ARTIGO]:
- O modelo gerado pela IA (Qwen 2.5) violou a propriedade fundamental de 'Soundness'.
- Existem erros lógicos graves de fluxo (ex: Deadlocks, loops infinitos ou Gateways órfãos).

Análise Comportamental:
- O modelo da IA bloqueia ou omite 16.7% dos caminhos legítimos do gabarito.

--- Resumo dos Resultados --- 


,Modelo,Fitness (Recall),Precision,Generalização,Simplicidade (Ref),Simplicidade (AI),Método
0,Qwen,83.33%,100.00%,35.21%,56.43%,56.76%,Token-Based Replay (Contagem de Fichas)
1,Llama,50.22%,100.00%,35.21%,56.43%,76.77%,Token-Based Replay (Contagem de Fichas)
2,Gemma,83.33%,100.00%,35.21%,56.43%,56.76%,Token-Based Replay (Contagem de Fichas)
